# Lab Day 10: Finetune Qwen3-VL-4B (LoRA) để định vị hư hại mặt đường
Mã nằm trong package `lab/` của repo (viết ở local, chạy ở đây). Notebook chỉ clone repo, gọi các lệnh `python -m lab.*` rồi chấm bằng `scripts/score.py`.

Thứ tự chạy: mục 0 → 1 → (2, 3 xem nhanh) → E0 → E1 → E2 → E3 → E4. Mỗi lần chạy lưu lên Drive.

## 0. Môi trường (Colab T4)

In [ ]:
# Runtime → Change runtime type → T4 GPU
!nvidia-smi -L
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
REPO = 'https://github.com/trungKoiKa/K4-Track-4-Day10-VisionLanguageModel.git'
OUT = '/content/drive/MyDrive/lab10'      # adapter, run.json, dự đoán, bằng chứng: lưu ở đây
os.makedirs(OUT, exist_ok=True)
if not os.path.exists('/content/repo'):
    !git clone $REPO /content/repo
%cd /content/repo
!git pull
!pip -q install -r requirements-colab.txt
!python -m pytest -q

In [ ]:
# Kaggle: dùng Colab Secrets, KHÔNG viết khoá vào notebook.
# Token mới: KAGGLE_API_TOKEN (KGAT_...). Khoá cũ: KAGGLE_USERNAME + KAGGLE_KEY.
import os
from google.colab import userdata
for k in ('KAGGLE_API_TOKEN', 'KAGGLE_USERNAME', 'KAGGLE_KEY'):
    try:
        os.environ[k] = userdata.get(k)
    except Exception:
        pass
print({k: k in os.environ for k in ('KAGGLE_API_TOKEN', 'KAGGLE_USERNAME', 'KAGGLE_KEY')})

## 1. Dữ liệu: tải và chia split

In [ ]:
!python scripts/prepare_data.py --kaggle
!cat data/splits/meta.json

In [ ]:
import json, collections
from scripts.labcommon import read_jsonl
splits = {s: read_jsonl(f'data/splits/{s}.jsonl') for s in ('train', 'val', 'test')}
for s, rows in splits.items():
    c = collections.Counter(o['label'] for r in rows for o in r['objects'])
    print(s, len(rows), 'ảnh', dict(c), len({r['session'] for r in rows}), 'phiên')
sess = {s: {r['session'] for r in rows} for s, rows in splits.items()}
assert not (sess['train'] & sess['val'] or sess['train'] & sess['test'] or sess['val'] & sess['test'])
print('không phiên nào xuất hiện ở hai split')

## 2. Nhãn: YOLO → JSON thang 0–1000

In [ ]:
from lab.labels import objects_to_answer, get_prompt, box_to_pixels
from lab.viz import draw
from IPython.display import display
for e in splits['train'][:6]:
    ans = objects_to_answer(e['objects'])
    print(e['image_id'], ans)
    # quy ngược về pixel và vẽ lại: box phải trùng box gốc
    pred = [{'label': b['label'], 'bbox': tuple(b['bbox_2d'])} for b in json.loads(ans)]
    display(draw(e, pred, 'kiểm tra nhãn', 'GT', 'data'))
print(get_prompt())

## 3. E0: zero-shot

In [ ]:
!python -m lab.infer --out work/e0.jsonl
!cp work/e0.jsonl $OUT/e0.jsonl

In [ ]:
!python scripts/score.py --pred work/e0.jsonl --name E0 --out report/evidence/e0.json
rows = read_jsonl('work/e0.jsonl')
for r in rows[:5]:
    print(repr(r['raw'][:300]))

## 4. Dựng dữ liệu huấn luyện: chat template và mask loss

In [ ]:
from lab.model import load_processor
from lab.dataset import RoadDataset, describe_example
proc = load_processor('Qwen/Qwen3-VL-4B-Instruct')
ex = RoadDataset('train', proc)[0]
print(describe_example(proc, ex))   # n_image_tokens phải là 220; supervised_text = JSON + <|im_end|>

## 5. Setup LoRA / đóng băng cho E1, E2, E3

In [ ]:
# apply_setup nằm trong lab/model.py và tự kiểm tra 252 (E1, E2) / 348 (E3) module LoRA.
# Smoke test 5 bước trước khi chạy thật (bắt lỗi sớm, tiết kiệm GPU):
!python -m lab.train --config configs/e3.yaml --out /content/smoke --set max_steps=5
!python -m lab.infer --run-dir /content/smoke --split val --limit 8 --out work/smoke.jsonl

## 6. Vòng huấn luyện

In [ ]:
# E1: LoRA LLM. Mỗi lệnh ~20–30 phút; adapter và run.json ghi thẳng lên Drive.
!python -m lab.train --config configs/e1.yaml --out $OUT/e1

In [ ]:
# E2: + projector (float32)
!python -m lab.train --config configs/e2.yaml --out $OUT/e2

In [ ]:
# E3: + LoRA ViT
!python -m lab.train --config configs/e3.yaml --out $OUT/e3

## 7. Suy luận và chấm E1–E3

In [ ]:
for n in ('e1', 'e2', 'e3'):
    !python -m lab.infer --run-dir $OUT/{n} --out work/{n}.jsonl
    !cp work/{n}.jsonl $OUT/{n}.jsonl
    !python -m lab.score_run --name {n.upper()} --pred work/{n}.jsonl --run-dir $OUT/{n} --out report/evidence/{n}.json

## 8. E4 (ablation tự chọn) và E5 (tuỳ chọn, GRPO)

In [ ]:
# E4 = rank. Đổi ĐÚNG MỘT biến so với setup tốt nhất (sửa BEST_CFG nếu E1/E2 tốt hơn E3).
BEST_CFG = 'e3'
for r in (4, 64):
    d = f'{OUT}/e4_rank_{r}'
    !python -m lab.train --config configs/{BEST_CFG}.yaml --out $d --set lora.r={r} --set lora.alpha={2*r}
    !python -m lab.infer --run-dir $d --out work/e4_rank_{r}.jsonl
    !python -m lab.score_run --name E4_rank_{r} --pred work/e4_rank_{r}.jsonl --run-dir $d --out report/evidence/e4_rank_{r}.json
# r=16 chính là setup gốc (E1–E3), dùng làm điểm so sánh thứ ba.

In [ ]:
# Ảnh minh hoạ (>= 6 ảnh đúng/sai) rồi gói kết quả để tải về máy
!python -m lab.viz --pred work/e3.jsonl --name E3 --out report/figures --n-ok 3 --n-bad 5
!zip -r $OUT/results.zip report/evidence report/figures data/splits/meta.json
!ls -la $OUT